# Ajustes de dados + pop-up de horários — Mobilidade Urbana — v2
- **C** Corrige `sentido_variante`: `Bairri`→`Bairro`, remove espaço de `" Terminal"`, e 115_B_DIR → `Bairro`.
- **D** Liga os horários da `160_T_GU` (sem linha no mapa) à linha `160_T_DIR2` (sem horários).
- **A** Pop-up com plano B: rota sem horários próprios mostra os horários da mesma linha nos outros sentidos, com aviso.
- **B** Cabeçalho fixo "Horários de dias úteis".
Backup de tudo antes de alterar. Saída: `Documents\Saidas_Notebooks\Ajustar_Popup_Mobilidade_v2.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração e conexão =====
import os, json, datetime, collections
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, Table
NB_NOME = "Ajustar_Popup_Mobilidade_v2"
URL_APUC = "https://geo.apucarana.pr.gov.br/portal"
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
HORARIOS_ID = "7b643a7396af425d820da4c55b06effe"
URL_LINHAS = "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0"
URL_HORARIOS = "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0"
TITULO_CAMADA = "Linhas de ônibus"
ROTA_ORIGEM, ROTA_DESTINO = "160_T_GU", "160_T_DIR2"

home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
CARIMBO = f"{datetime.datetime.now():%Y%m%d_%H%M%S}"
PASTA_BKP = os.path.join(PASTA_SAIDA, f"backup_mobilidade_{CARIMBO}"); os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
def salvar(nome, obj):
    p = os.path.join(PASTA_BKP, nome)
    with open(p, "w", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    return p

log("▶ CÉLULA 1 — Configuração e conexão: iniciando...")
gis = GIS("pro")
fl_lin = FeatureLayer(URL_LINHAS, gis)
tb_hor = Table(URL_HORARIOS, gis)
log("  Conectado como:", gis.users.me.username, "|", gis.url)
log("  Pasta de backup:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Backups =====
log("\n▶ CÉLULA 2 — Backups: iniciando...")
oid_lin = fl_lin.properties.objectIdField
oid_hor = tb_hor.properties.objectIdField
lin = [f.attributes for f in fl_lin.query(where="1=1", out_fields="*", return_geometry=False).features]
hor = [f.attributes for f in tb_hor.query(where="1=1", out_fields="*", return_geometry=False).features]
log("  Linhas (atributos):", salvar("linhas_atributos.json", lin), f"({len(lin)})")
log("  Horários:", salvar("horarios.json", hor), f"({len(hor)})")
wm_item = gis.content.get(WEBMAP_ID)
wm = wm_item.get_data()
log("  Web map:", salvar("webmap.json", wm))
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — (C) Correções em sentido_variante =====
log("\n▶ CÉLULA 3 — Correções na camada de linhas: iniciando...")
upd = []
for a in lin:
    v = a.get("sentido_variante")
    novo = v
    if v is not None and v.strip() != v:
        novo = v.strip()
    if (novo or "") == "Bairri":
        novo = "Bairro"
    if a.get("id_rota") == "115_B_DIR" and novo != "Bairro":
        novo = "Bairro"
    if novo != v:
        upd.append({"attributes": {oid_lin: a[oid_lin], "sentido_variante": novo}})
        log(f"  · {a['id_rota']}: '{v}' → '{novo}'")
if upd:
    r = fl_lin.edit_features(updates=upd)
    ok = sum(1 for x in r.get("updateResults", []) if x.get("success"))
    log(f"  Atualizadas: {ok}/{len(upd)}")
else:
    log("  Nada a corrigir (já estava corrigido).")
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — (D) Religar horários 160_T_GU → 160_T_DIR2 =====
log("\n▶ CÉLULA 4 — Religação dos horários da linha 160: iniciando...")
rotas_lin = {a["id_rota"] for a in lin}
cont = collections.Counter(a["id_rota"] for a in hor)
if ROTA_DESTINO not in rotas_lin:
    log(f"  ! {ROTA_DESTINO} não existe na camada de linhas — nada feito.")
elif cont.get(ROTA_DESTINO, 0) > 0:
    log(f"  ! {ROTA_DESTINO} já tem {cont[ROTA_DESTINO]} horários — nada feito (evita misturar).")
elif cont.get(ROTA_ORIGEM, 0) == 0:
    log(f"  {ROTA_ORIGEM} não tem horários (provavelmente já religado antes) — nada feito.")
else:
    upd = [{"attributes": {oid_hor: a[oid_hor], "id_rota": ROTA_DESTINO}} for a in hor if a["id_rota"] == ROTA_ORIGEM]
    r = tb_hor.edit_features(updates=upd)
    ok = sum(1 for x in r.get("updateResults", []) if x.get("success"))
    log(f"  {ok}/{len(upd)} horários passaram de {ROTA_ORIGEM} para {ROTA_DESTINO}.")
log("✔ CÉLULA 4 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 5 — (A + B) Novo pop-up de horários =====
log("\n▶ CÉLULA 5 — Atualização do pop-up: iniciando...")
ARCADE = """
var tab = FeatureSetByPortalItem(Portal('%s'), '%s', 0, ['id_rota','sentido','ponto','horario'], false);
var rota = $feature.id_rota;

function chips(fs) {
  var out = '';
  var pontoAnt = '#';
  for (var h in OrderBy(fs, 'ponto ASC, horario ASC')) {
    var p = IIf(IsEmpty(h.ponto), '', Text(h.ponto));
    if (p != pontoAnt) {
      if (pontoAnt != '#') { out += '</div>'; }
      if (p != '') { out += '<div style="font-size:11px;color:#555;margin:6px 0 2px">&#128205; Saída: ' + p + '</div>'; }
      out += '<div>';
      pontoAnt = p;
    }
    out += '<span style="display:inline-block;margin:2px;padding:3px 8px;border-radius:4px;background:#1A2942;color:#fff;font-weight:bold">' + Left(Text(h.horario), 5) + '</span>';
  }
  if (pontoAnt != '#') { out += '</div>'; }
  return out;
}

var html = '<div style="font-family:Arial,sans-serif;font-size:13px">';
html += '<div style="margin:10px 0 4px;font-weight:bold;color:#1A2942;border-bottom:2px solid #F5A623">&#128197; Horários de dias úteis</div>';

var hs = Filter(tab, 'id_rota = @rota');
if (Count(hs) > 0) {
  html += chips(hs);
} else {
  var linha = Split(rota, '_')[0];
  var pref = linha + '_%%';
  var alt = Filter(tab, 'id_rota LIKE @pref');
  if (Count(alt) == 0) {
    html += '<p style="color:#888;margin:4px 0">Sem horários cadastrados para esta linha.</p>';
  } else {
    html += '<div style="background:#FFF4E0;border-left:4px solid #F5A623;padding:6px 8px;margin:6px 0;font-size:12px;color:#5a4300">'
          + 'Este sentido não tem horários próprios. Abaixo, os horários de saída da linha ' + linha + ' nos outros sentidos.</div>';
    for (var s in Distinct(alt, 'sentido')) {
      var sv = s.sentido;
      var sub = IIf(IsEmpty(sv), Filter(alt, 'sentido IS NULL'), Filter(alt, 'sentido = @sv'));
      html += '<div style="margin:8px 0 2px;font-weight:bold;color:#4285F4;font-size:12px">' + IIf(IsEmpty(sv), 'Outro sentido', sv) + '</div>';
      html += chips(sub);
    }
  }
}
html += '</div>';
return { type: 'text', text: html };
""" % (URL_APUC, HORARIOS_ID)

alvo = [ol for ol in wm.get("operationalLayers", []) if ol.get("title") == TITULO_CAMADA]
if not alvo:
    raise RuntimeError(f"Camada '{TITULO_CAMADA}' não encontrada no web map.")
pop = alvo[0].setdefault("popupInfo", {})
info = {"name": "horarios", "title": "Horários", "expression": ARCADE, "returnType": "dictionary"}
pop["expressionInfos"] = [info]
pop["popupElements"] = [{"type": "fields"}, {"type": "expression", "expressionInfo": info}]
wm_item.update(item_properties={"text": json.dumps(wm)})
log("  Pop-up atualizado: cabeçalho 'Horários de dias úteis' + plano B por linha.")
log("✔ CÉLULA 5 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 6 — Conferência final + links =====
log("\n▶ CÉLULA 6 — Conferência final: iniciando...")
lin2 = [f.attributes for f in fl_lin.query(where="1=1", out_fields="id_rota,sentido_variante", return_geometry=False).features]
hor2 = [f.attributes for f in tb_hor.query(where="1=1", out_fields="id_rota", return_geometry=False).features]
c2 = collections.Counter(a["id_rota"] for a in hor2)
rot2 = {a["id_rota"] for a in lin2}
sem = sorted(a["id_rota"] for a in lin2 if c2.get(a["id_rota"], 0) == 0)
linhas_com = {r.split("_")[0] for r in c2}
sem_nada = [r for r in sem if r.split("_")[0] not in linhas_com]
log(f"  Rotas sem horários próprios: {len(sem)} (antes: 30) → cobertas pelo plano B: {len(sem)-len(sem_nada)}")
log(f"  Rotas cuja LINHA inteira não tem nenhum horário: {len(sem_nada)} {sem_nada}")
log(f"  Horários órfãos (sem linha no mapa): {sorted(set(c2) - rot2)}")
log("  sentido_variante agora:", dict(collections.Counter(a["sentido_variante"] for a in lin2)))
log("\n  Web map (Map Viewer):", f"{URL_APUC}/apps/mapviewer/index.html?webmap={WEBMAP_ID}")
log("  Item do web map:", f"{URL_APUC}/home/item.html?id={WEBMAP_ID}")
log("  Teste: Ctrl+F5 no Map Viewer → clique na linha 320 (sentido Bairro) e numa linha 160.")
log(f"  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 6 concluída com sucesso.")
